# 09. Machine learning I — klasifikasi angka tulisan tangan

**Capaian:** Melatih baseline, membandingkan vektor piksel vs HOG, dan menilai kesalahan.

**Prasyarat:** Python dasar, NumPy, dan membaca array. **Estimasi:** 75–100 menit.

> Kerjakan dari atas ke bawah. Setiap grafik wajib disertai penafsiran, bukan hanya ditempel.

## Konsep dan model

Masalah: klasifikasi $10$ kelas dari citra $8\times8$. Kita membagi data secara *stratified*, melatih hanya pada train, lalu menguji pada test yang belum terlihat. Metrik utama: macro-F1 dan confusion matrix. Dataset tersedia di scikit-learn, tidak mengklaim mewakili tulisan tangan dari populasi lain.

## Data dan sampel

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
digits=load_digits(); X=digits.data; y=digits.target
print('Sampel:',len(y),'dimensi:',digits.images.shape,'kelas:',np.unique(y))
fig,ax=plt.subplots(1,8,figsize=(12,2))
for a,i in zip(ax,range(8)):
    a.imshow(digits.images[i],cmap='gray');a.set_title(str(y[i]));a.axis('off')
plt.show()

## Split dan baseline linear

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
tr,te=train_test_split(np.arange(len(y)),test_size=0.25,stratify=y,random_state=42)
baseline=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1500,random_state=42))
baseline.fit(X[tr],y[tr]); p0=baseline.predict(X[te])
print(classification_report(y[te],p0,digits=3,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y[te],p0,values_format='d',cmap='Blues');plt.show()

## HOG + SVM sebagai pembanding

In [ ]:
from skimage.feature import hog
from sklearn.svm import SVC
X_hog=np.stack([hog(im,orientations=8,pixels_per_cell=(4,4),cells_per_block=(1,1)) for im in digits.images])
svm=make_pipeline(StandardScaler(),SVC(kernel='rbf',C=4,gamma='scale'))
svm.fit(X_hog[tr],y[tr]);p1=svm.predict(X_hog[te])
from sklearn.metrics import f1_score
print('Macro-F1 piksel:',round(f1_score(y[te],p0,average='macro'),3))
print('Macro-F1 HOG+SVM:',round(f1_score(y[te],p1,average='macro'),3))

## Analisis salah prediksi

In [ ]:
wrong=te[p1!=y[te]]
print('Jumlah kesalahan HOG+SVM:',len(wrong))
fig,ax=plt.subplots(1,min(6,len(wrong)),figsize=(10,2)) if len(wrong)>1 else (None,[plt.gca()])
for a,i in zip(np.ravel(ax),wrong[:6]):
    a.imshow(digits.images[i],cmap='gray');a.set_title(f'Benar {y[i]}; pred {svm.predict(X_hog[i:i+1])[0]}');a.axis('off')
plt.show()

## Diskusi, HOTS, dan latihan

1. Mana lebih baik: baseline piksel atau HOG+SVM? Jangan memilih berdasarkan asumsi.
2. Uji rotasi ±10° pada test (tanpa mengubah train) dan amati perubahan F1.
3. Mengapa tuning hiperparameter harus memakai validation atau cross-validation pada train, **bukan test**?

**Laporan singkat:** rumusan masalah → rancangan eksperimen → hasil → interpretasi → keterbatasan → kesimpulan.